# Perform initial VCF filtering

In [109]:
import gzip
from collections import namedtuple
from collections import Counter
import numpy as np

In [10]:
joint_vcf = "../../../palladium-vcf/10_trgt/output/merged.vcf.gz"

In [101]:
TrAllele = namedtuple("TrAllele", "seq purity meth")
TrRec = namedtuple("TrRec", "trid motifs gts")


def get_vcf_rec(vcf_path):
    with gzip.open(vcf_path, "rb") as file:
        for line in file:
            line = line.decode("ascii")
            sl = line.split()
            if line.startswith("#CHROM"):
                index = sl.index("FORMAT") + 1
                samples = sl[index:]
                continue
            if line[0] == "#":
                continue
            trid = sl[7].split(";")[0].replace("TRID=", "")
            motifs = sl[7].split(";")[-2].replace("MOTIFS=", "").split()
            alleles = [sl[3]]  # Start with the ref allele
            alleles.extend(sl[4].split(","))  # And then extend with alts
            alleles = [a[1:] for a in alleles] # Remove padding base
            gts = {sample: [] for sample in samples}
            for sample, rec in zip(samples, sl[9:]):
                rec = rec.split(":")
                al_idxs = [int(idx) if idx != "." else None for idx in rec[0].split("/")]
                aps = [float(val) if val != "." else None for val in rec[-2].split(",")]
                ams = [float(val) if val != "." else None for val in rec[-1].split(",")]
                for al_idx, ap, am in zip(al_idxs, aps, ams):
                    if al_idx is None:
                        break
                    seq = alleles[al_idx]
                    allele = TrAllele(seq, ap, am)
                    gts[sample].append(allele)
            yield TrRec(trid, motifs, gts)

In [113]:
filtered_trs = []
total_trs = 0
num_filtered = 0

for index, rec in enumerate(get_vcf_rec(joint_vcf)):
    all_alleles = []
    all_purities = []
    for sample, alleles in rec.gts.items():
        all_alleles.extend([a.seq for a in alleles])
        all_purities.extend([a.purity for a in alleles if a.purity])

    total_trs += 1

    # Skip repeats with at least one missing genotype 
    if len([gt for sample, gt in rec.gts.items() if gt]) != 10:
        continue

    # Skip repeats where at least one allele missing purity
    if len(all_alleles) != len(all_purities):
        continue

    # Skip repeats with very low purity
    if np.mean(all_purities) < 0.5:
        continue

    # Skip repeats with singleton alleles since these are
    # enriched in errors or de novos
    count_by_allele = Counter(all_alleles)
    min_count = min(count_by_allele.values())
    if min_count == 1:
        continue

    # Skip non-polymorphic repeats
    if len(count_by_allele) == 1:
        continue
    
    # Skip repeats with very short alleles
    longest_allele = max(len(a) for a in all_alleles)
    if longest_allele < 10:
        continue
    
    filtered_trs.append(rec.trid)
    num_filtered += 1

print(f"Num trs processed = {total_trs}; left after filtering = {num_filtered}")

Num trs processed = 7722730; left after filtering = 650610


In [112]:
# Num trs processed = 7717950; left after filtering = 655692